In [1]:
!pip install scikit-uplift

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


In [1]:
from sklift.datasets import fetch_lenta

dataset = fetch_lenta()
data, target, treatment, DESCR, feature_names, target_name, treatment_name  = dataset.data, dataset.target, dataset.treatment, dataset.DESCR, dataset.feature_names, dataset.target_name, dataset.treatment_name

# alternative option
# data, target, treatment = fetch_lenta(return_X_y_t=True)

In [2]:
data.describe()

,age,cheque_count_12m_g20,cheque_count_12m_g21,cheque_count_12m_g25,cheque_count_12m_g32,cheque_count_12m_g33,cheque_count_12m_g38,cheque_count_12m_g39,cheque_count_12m_g41,cheque_count_12m_g42,...,sale_sum_6m_g24,sale_sum_6m_g25,sale_sum_6m_g26,sale_sum_6m_g32,sale_sum_6m_g33,sale_sum_6m_g44,sale_sum_6m_g54,stdev_days_between_visits_15d,stdev_discount_depth_15d,stdev_discount_depth_1m
count,675264.000000,687029.000000,687029.000000,687029.000000,687029.000000,687029.000000,687029.000000,687029.000000,687029.000000,687029.00000,...,683914.000000,683914.000000,683914.000000,683914.000000,683914.000000,683914.000000,683914.000000,610075.000000,546576.00000,546661.000000
mean,43.740718,2.567841,3.523936,5.702353,2.913084,5.370441,4.766505,2.215738,5.508804,3.64978,...,738.580018,218.296431,340.566110,338.279236,790.497073,431.775305,400.094652,0.291046,0.06995,0.112960
std,14.842062,5.405423,6.919543,11.000989,5.999545,9.447374,8.244424,4.168154,9.033987,5.90708,...,2105.583811,523.454723,865.868461,919.501039,1864.715986,986.999892,911.842583,0.909622,0.12657,0.142271
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
25%,32.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,1.000000,0.00000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
50%,42.000000,1.000000,1.000000,2.000000,1.000000,2.000000,2.000000,1.000000,3.000000,2.00000,...,219.700000,62.000000,76.390000,0.000000,219.990000,138.990000,132.690000,0.000000,0.00000,0.000000
75%,56.000000,3.000000,4.000000,6.000000,3.000000,6.000000,6.000000,3.000000,7.000000,5.00000,...,775.120000,230.520000,353.067500,310.580000,833.785000,502.820000,436.590000,0.000000,0.10150,0.225700
max,100.000000,1556.000000,1763.000000,2666.000000,1151.000000,2199.000000,2046.000000,964.000000,1819.000000,1241.00000,...,505188.120000,177758.220000,283892.600000,208722.300000,561862.500000,232231.450000,235498.160000,9.192400,0.70550,0.706600


In [8]:
data[treatment_name] = treatment
data[target_name] = target

In [9]:
data.shape

(687029, 195)

In [11]:
data.to_csv('D:/PostPhD/code/uplift_res/uplift_lab/data/lenta_data.csv', index=False)

In [31]:
import pandas as pd
import numpy as np

def process_dataframe(data):
    """
    处理DataFrame：
    1. 对gender列进行编码：'Ж' -> 1, 'М' -> 0
    2. 对group列进行编码：'test' -> 1, 'control' -> 0
    3. 删除缺失值比率大于40%的列
    4. 删除包含缺失值的行
    """
    
    # 复制原始数据，避免修改原DataFrame
    df = data.copy()
    
    # 1. 对gender列进行编码
    gender_mapping = {'Ж': 1, 'М': 0}
    df['gender'] = df['gender'].map(gender_mapping)
    
    # 2. 对group列进行编码
    group_mapping = {'test': 1, 'control': 0}
    df['group'] = df['group'].map(group_mapping)
    
    # 3. 计算各列的缺失值比率
    missing_ratio = df.isnull().sum() / len(df)
    
    # 4. 删除缺失值比率大于40%的列
    columns_to_keep = missing_ratio[missing_ratio <= 0.2].index
    df = df[columns_to_keep]
    
    # 5. 删除包含缺失值的行
    df_cleaned = df.dropna()
    
    return df_cleaned

# 使用示例
# 假设你有一个名为data的DataFrame
processed_data = process_dataframe(data)

In [32]:
processed_data.shape

(175774, 110)

In [35]:
processed_data[['gender','age']].head()

,gender,age
0,1.0,47.0
3,1.0,65.0
4,1.0,61.0
5,1.0,27.0
8,1.0,69.0


In [36]:
processed_data.to_csv('D:/PostPhD/code/uplift_res/uplift_lab/data/lenta_data_processed.csv', index=False)

In [33]:
processed_data[['group','response_att']].groupby('group').mean()

,response_att
group,
0,0.191848
1,0.201887


In [34]:
processed_data[['group','response_att']].head()

,group,response_att
0,1,0
3,1,0
4,1,0
5,1,0
8,1,0


In [39]:
import pandas as pd
import argparse
import os


def stratified_sample(input_path='data/lenta_data_processed.csv', output_path='data/lenta_data_sampled_30k_per_group.csv',
                      group_col='group', n_per_group=30000, random_state=None):
    df = pd.read_csv(input_path)
    if group_col not in df.columns:
        raise ValueError(f"Expected group column '{group_col}' in dataframe")

    samples = []
    counts = {}
    for grp, gdf in df.groupby(group_col):
        size = len(gdf)
        take = min(n_per_group, size)
        if take < n_per_group:
            print(f"Warning: group={grp} has only {size} rows; taking all {take} rows")
        sampled = gdf.sample(n=take, replace=False, random_state=random_state)
        samples.append(sampled)
        counts[grp] = (take, size)

    if not samples:
        raise ValueError("No groups found to sample from")

    result = pd.concat(samples, ignore_index=True)
    # shuffle combined result
    result = result.sample(frac=1, random_state=random_state).reset_index(drop=True)

    out_dir = os.path.dirname(output_path)
    if out_dir and not os.path.exists(out_dir):
        os.makedirs(out_dir, exist_ok=True)

    result.to_csv(output_path, index=False)

    print('\nSampling summary:')
    total_taken = 0
    for grp, (taken, size) in counts.items():
        print(f"Group={grp}: taken={taken}, available={size}")
        total_taken += taken
    print(f"Total rows in sample: {total_taken}. Saved to {output_path}")

    return result


if __name__ == '__main__':
    args = {
        'input': './data/lenta_data_processed.csv',
        'output': './data/lenta_data_sampled_30k_per_group.csv',
        'group': 'group',
        'n': 30000,
        'seed': 2026
    }
    stratified_sample(input_path=args['input'], output_path=args['output'], group_col=args['group'], n_per_group=args['n'], random_state=args['seed'])



Sampling summary:
Group=0: taken=30000, available=43472
Group=1: taken=30000, available=132302
Total rows in sample: 60000. Saved to ./data/lenta_data_sampled_30k_per_group.csv


In [40]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import chi2_contingency, fisher_exact

def load_data(path='./data/lenta_data_processed.csv'):
    df = pd.read_csv(path)
    return df


def run_tests(df):
    if 'group' not in df.columns or 'response_att' not in df.columns:
        raise ValueError("Expected columns 'group' and 'response_att' in the dataframe")

    # Ensure binary values are 0/1
    ct = pd.crosstab(df['group'], df['response_att'])
    print('\nContingency table (rows=group, cols=response_att):')
    print(ct)

    # Compute proportions for response_att == 1
    if 1 in ct.columns:
        success = ct[1].values
    else:
        success = [0] * ct.shape[0]
    n = ct.sum(axis=1).values

    props = success / n
    for grp, s, nn, p in zip(ct.index, success, n, props):
        print(f"Group={grp}: successes={s}, n={nn}, proportion={p:.4f}")

    # If exactly two groups, run two-proportion z-test and Fisher/Chi2
    if len(success) == 2:
        stat, pval = proportions_ztest(success, n)
        print(f"\nTwo-proportion z-test: stat={stat:.4f}, p-value={pval:.4e}")

    # Chi-square test (works for any RxC)
    chi2, p_chi, dof, expected = chi2_contingency(ct)
    print(f"Chi-square test: chi2={chi2:.4f}, p-value={p_chi:.4e}, dof={dof}")

    # If 2x2, also show Fisher exact
    if ct.shape == (2, 2):
        # fisher_exact expects [[a, b], [c, d]] where a is success in group0
        oddsratio, p_fisher = fisher_exact(ct.values)
        print(f"Fisher exact test: oddsratio={oddsratio:.4f}, p-value={p_fisher:.4e}")


if __name__ == '__main__':
    path = './data/lenta_data_sampled_30k_per_group.csv' # lenta_data_processed lenta_data lenta_data_sampled_30k_per_group
    # df = load_data()
    df = load_data(path=path)
    print('Loaded', len(df), 'rows from ', path)
    run_tests(df)


Loaded 60000 rows from  ./data/lenta_data_sampled_30k_per_group.csv

Contingency table (rows=group, cols=response_att):
response_att      0     1
group                    
0             24274  5726
1             23798  6202
Group=0: successes=5726, n=30000, proportion=0.1909
Group=1: successes=6202, n=30000, proportion=0.2067

Two-proportion z-test: stat=-4.8691, p-value=1.1208e-06
Chi-square test: chi2=23.6091, p-value=1.1803e-06, dof=1
Fisher exact test: oddsratio=1.1048, p-value=1.1773e-06


In [21]:
DESCR

'Lenta Uplift Modeling Dataset\n================================\n\nData description\n################\n\nAn uplift modeling dataset containing data about Lenta\'s customers grociery shopping and related marketing campaigns.\n\nSource: **BigTarget Hackathon** hosted by Lenta and Microsoft in summer 2020.\n\nFields\n################\n\nMajor features:\n\n    * ``group`` (str): treatment/control group flag\n    * ``response_att`` (binary): target\n    * ``gender`` (str): customer gender\n    * ``age`` (float): customer age\n    * ``main_format`` (int): store type (1 - grociery store, 0 - superstore)\n\n\n.. list-table::\n    :align: center\n    :header-rows: 1\n    :widths: 5 5\n\n    * - Feature\n      - Description\n    * - CardHolder\n      - customer id\n    * - customer\n      - age\n    * - children\n      - number of children\n    * - cheque_count_[3,6,12]m_g*\n      - number of customer receipts collected within last 3, 6, 12 months\n        before campaign. g* is a product group

In [11]:
len(feature_names), target_name, treatment_name

(193, 'response_att', 'group')

In [14]:
import pandas as pd
sourceFile = './data/archive/sourceData.csv'

In [15]:
sourceData = pd.read_csv(sourceFile)[feature_names + [target_name, treatment_name]]

In [25]:
feature_names

['age',
 'cheque_count_12m_g20',
 'cheque_count_12m_g21',
 'cheque_count_12m_g25',
 'cheque_count_12m_g32',
 'cheque_count_12m_g33',
 'cheque_count_12m_g38',
 'cheque_count_12m_g39',
 'cheque_count_12m_g41',
 'cheque_count_12m_g42',
 'cheque_count_12m_g45',
 'cheque_count_12m_g46',
 'cheque_count_12m_g48',
 'cheque_count_12m_g52',
 'cheque_count_12m_g56',
 'cheque_count_12m_g57',
 'cheque_count_12m_g58',
 'cheque_count_12m_g79',
 'cheque_count_3m_g20',
 'cheque_count_3m_g21',
 'cheque_count_3m_g25',
 'cheque_count_3m_g42',
 'cheque_count_3m_g45',
 'cheque_count_3m_g52',
 'cheque_count_3m_g56',
 'cheque_count_3m_g57',
 'cheque_count_3m_g79',
 'cheque_count_6m_g20',
 'cheque_count_6m_g21',
 'cheque_count_6m_g25',
 'cheque_count_6m_g32',
 'cheque_count_6m_g33',
 'cheque_count_6m_g38',
 'cheque_count_6m_g39',
 'cheque_count_6m_g40',
 'cheque_count_6m_g41',
 'cheque_count_6m_g42',
 'cheque_count_6m_g45',
 'cheque_count_6m_g46',
 'cheque_count_6m_g48',
 'cheque_count_6m_g52',
 'cheque_count_